In [42]:
from dotenv import load_dotenv
from google import genai
from google.genai import types
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from qdrant_client import models, QdrantClient
from pathlib import Path
from IPython.display import Markdown, display
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
import os

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY_NEW")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_URL_ENDPOINT = os.getenv("QDRANT_URL_ENDPOINT")

In [40]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
GOOGLE_GEMINI_MODEL2 = os.getenv("GOOGLE_GEMINI_MODEL2")
encoder = SentenceTransformer(model_name)

In [43]:
client = QdrantClient(
    api_key=QDRANT_API_KEY,
    url = QDRANT_URL_ENDPOINT,
)
google_client = OpenAI(
    api_key=GOOGLE_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [22]:
if not client.collection_exists("my_first_cluster"):
    client.create_collection(
        collection_name="my_first_cluster",
        vectors_config=models.VectorParams(
            size = encoder.get_sentence_embedding_dimension(),
            distance=models.Distance.COSINE,
        )
    )

In [23]:
def get_all_user_data(dir_name : str = "knowledge-base"):
    content = ""
    dir_path = Path(dir_name)
    for file in dir_path.rglob("*"):
        if file.is_file():
            content += file.read_text(encoding="utf-8")
            content += "\n\n"
    return content

In [24]:
from langchain_core.documents.base import Document
headers_to_spilt_on = [
    ("#", "title"),
    ("##", "Subtitle"),
    ("###", "Section"),
]
def split_markdown_text(content : str) -> Document:
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_spilt_on,
    )
    return markdown_splitter.split_text(content)

def split_docs_using_recursive_text_splitter(documents : list[Document]):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 800,
        chunk_overlap = 200,
    )
    return text_splitter.split_documents(documents)
    

In [25]:
def split_company_content(content : str):
    markdown_docs = split_markdown_text(content)
    all_chunks = []
    for doc in markdown_docs:
        small_chunk = split_docs_using_recursive_text_splitter([doc])
        all_chunks.extend(small_chunk)
    return all_chunks

In [26]:
docs = split_company_content(get_all_user_data())
len(docs)
docs[0]

Document(metadata={'title': 'About Insurellm'}, page_content='Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.  \nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.')

In [27]:
embedding_vectors = [encoder.encode(doc.page_content).tolist() for doc in docs]

# Visualization

In [ ]:
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

# 1) Prepare vectors (embedding_vectors is already a list of lists)
X = np.array(embedding_vectors)

# 2) Reduce 768-dim → 2D
tsne = TSNE(n_components=2, random_state=42, perplexity=30, init="random")
reduced_vectors = tsne.fit_transform(X)

# 3) Prepare visualization metadata
# doc.page_content → text
# doc.metadata → dict
hover_texts = [
    f"<b>Title:</b> {doc.metadata.get('title', 'N/A')}<br>"
    f"<b>Section:</b> {doc.metadata.get('section', 'N/A')}<br>"
    f"<b>Preview:</b> {doc.page_content[:200]}..."
    for doc in docs
]

# Optional: color points by document title or section
colors = [
    hash(doc.metadata.get("title", "default")) % 15   # gives 12 color groups
    for doc in docs
]

# 4) Build the plot
fig = go.Figure(
    data=[
        go.Scatter(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            mode="markers",
            marker=dict(
                size=6,
                color=colors,
                opacity=0.85,
                colorscale="Viridis"
            ),
            text=hover_texts,
            hoverinfo="text"
        )
    ]
)

fig.update_layout(
    title="2D t-SNE Visualization of Your Qdrant Knowledge Base",
    width=900,
    height=650,
    xaxis_title="TSNE Component 1",
    yaxis_title="TSNE Component 2",
    margin=dict(r=20, b=20, l=20, t=60)
)


In [34]:
import uuid
def upload_to_qdrant():
    return client.upsert(
        collection_name="my_first_cluster",
        points = [
            models.PointStruct(
                id = str(uuid.uuid4()),
                payload = {
                    "metadata" : doc.metadata,
                    "page_content" : doc.page_content
                },
                vector=vector
            )
            for doc, vector in zip(docs, embedding_vectors)
        ]
    )

In [61]:
def get_relevant_context(user_query : str, limit: int = 10):
    query_vector = encoder.encode(user_query).tolist()
    hits = client.query_points(
        collection_name="my_first_cluster",
        query=query_vector,
        limit=limit
    ).points
    relevent_context_list = [hit.payload["page_content"] for hit in hits if hit.score > 0.45]
    return '\n'.join(relevent_context_list)

In [62]:
hits = get_relevant_context("what is Carllm?")

In [63]:
display(Markdown(hits))

Carllm is an innovative auto insurance product developed by Insurellm, designed to streamline the way insurance companies offer coverage to their customers. Powered by cutting-edge artificial intelligence, Carllm utilizes advanced algorithms to deliver personalized auto insurance solutions, ensuring optimal coverage while minimizing costs. With a robust infrastructure that supports both B2B and B2C customers, Carllm redefines the auto insurance landscape and empowers insurance providers to enhance customer satisfaction and retention.
- Increase partnerships with automakers for integrated insurance solutions.
- Enhance the **AI customer support system** to include multi-language support.  
Carllm is not just an auto insurance product; it is a transformative tool for the insurance industry. Join us on this exciting journey as we redefine the future of auto insurance with technology and customer-centric solutions.
- **Carllm** - Auto insurance platform for insurers
- **Homellm** - Home insurance platform for insurers
- **Lifellm** - Life insurance platform with AI-powered underwriting
- **Healthllm** - Comprehensive health insurance platform
- **Bizllm** - Commercial insurance platform for business coverage

In [69]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledge, friendly assistent representing the company Insurellm.
You have to give precious and accurate answer to the user's question.
If relevant, use the given context to answer the question.
If you don't know the answer, say so.
Relevant Contex:
{context}
"""

In [72]:
def answer_question(user_query : str, history):
    additional_context = get_relevant_context(user_query)
    system_instruction = SYSTEM_PROMPT_TEMPLATE.format(context = additional_context)
    messages = [{'role' : 'system', 'content' : system_instruction}] + history + [{'role' : 'user', 'content' : user_query}]
    response = google_client.chat.completions.create(model=GOOGLE_GEMINI_MODEL2, messages=messages)
    return response.choices[0].message.content


In [73]:
import gradio as gr
view = gr.ChatInterface(answer_question, type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
